# Generating web-text summaries with LM for classification fine-tuning
### **Objectives**
Generic web-scraping of business websites often capture website text data in "nonsensical" form. For example, business websites often contain many headings which gives rise to incoherent sequences. Furthermore, some HTML tags like the span tag might also contain meaningless text sequences.

*Extractive summarisation is the strategy of concatenating extracts taken from a text into a summary, while abstractive summarisation involves paraphrasing the corpus using novel sentences. Most of the summarisation models are based on models that generate novel text (they are Natural Language Generation models, like, for example, GPT-3). This means that the summarisation models will also generate novel text, which makes them abstractive summarisation models.*

***Remember, garbage in, garbage out~ The quality of input text data for model fine-tuning is extremely crucial, period.***

### **Explore solutions:**
- One potential solution is the implementation of prompt tuning in Language Models (LLMs) to summarize incoherent text sequences obtained from the generic web scraping of these business websites.
- With access to a "more powerful" LM like GPT-4, coupled with good prompting techniques, one might consider the direct generation of business indicators (labels) for business profiling.

### **Methodology**
- Summarize all text in dataset to coherent form, highlighting business websites' features for classification of hiring firms (current use-case)
- Review samples to check for coherency.


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df_path = 'data/web_text.csv'
df = pd.read_csv(df_path)

Mounted at /content/drive


In [ ]:
# Bin word counts
def bin_word_count(count):
    if count <= 500:
        return "Short text"
    elif 500 < count <= 1000:
        return "Medium text"
    else:
        return "Long text"

# Apply the binning function to the 'word_counts' column
df['text_class'] = df['word_count'].apply(bin_word_count)

In [ ]:
df.head()

,index,text,innovative_label,word_count,text_class
0,192,Expert Web Development and Digital Marketing i...,0,296,Short text
1,3690,Home Oon Bazul Skip to content Search for Home...,0,1374,Long text
2,3356,Our Latest Projects Twenty20 Energy Home About...,0,1093,Long text
3,2009,Literature Library Literature Library InVivos ...,0,444,Short text
4,1514,Aviation Unitech About Unitech Our Story Certi...,0,248,Short text


## Summary algorithm
- Test LM's ability to summarize 10 random samples of "Long text"
- Implement text summarization to generate new column "text_summary"
- Experimentation: Prompt tuning to get best summarization prompt - Future works

In [ ]:
# Sample 10 random long texts from df (for testing)
test = df[df['text_class'] == 'Long text'].sample(10)

In [ ]:
# import dependencies
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm.notebook import tqdm

In [ ]:
# set device and max tokens model can generate
device = "cuda" if torch.cuda.is_available() else "cpu"
_new_max_tokens = 32 if not torch.cuda.is_available() else 120

# Print device information and token limit
print(f"Device: {device}")
print(f"Max New Tokens: {_new_max_tokens}")

Device: cuda
Max New Tokens: 120


In [ ]:
model_name = "Qwen/Qwen1.5-0.5B-Chat"
# model_name = "microsoft/Phi-3-mini-4k-instruct" ## To try.

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set up chat-based text completion function with Qwen LM
def complete(messages, max_new_tokens=_new_max_tokens):

  # we first want to convert messages to text
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  # tokenize text
  inputs = tokenizer(text, return_tensors="pt").to(device)

  # Generate output - model to predict next tokens from input tokens
  outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

  # extract generated tokens from chunk of input tokens and generated tokens
  generated_ids = outputs[0][len(inputs.input_ids[0]):]

  # utilize tokenizer to decode model's response
  return text, tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
# Function to test outputs < 100 words
def is_under_100_words(text):
  # test if the summary has at most 100 words
  return len(text.split()) <= 100

# Perform summarization of long text and review results
num_passed = 0
for webtext in tqdm(test["text"]):

  # Construct prompt with given AI role and task instruction (with review text)
  messages = [
      {"role": "Act as a professional text summarizer",
       "content": f"Summarize website text in English and in strictly less than 100 words: {webtext}"}
  ]
  _, completion = complete(messages)
  print("OUTPUT:")
  print()
  print(completion)

  passed = is_under_100_words(completion)
  if passed:
    num_passed += 1

  print(f"TEST PASSED: {passed}")
  print(f"WORD COUNT: {len(completion.split())}")
  print()

print("=" * 100)
print()

print(f"PASSED: {num_passed}/{len(test)}")

  0%|          | 0/10 [00:00<?, ?it/s]

OUTPUT:

Roquette is a global leader in providing carbohydrates for cell culture and developing novel technology for protein stabilization. They offer a wide range of products, including protein stability, precision dispense, and virtual lab services. Roquette has partnerships with companies such as Biopharma,药房, and cosmetics.
TEST PASSED: True
WORD COUNT: 44

OUTPUT:

"Workplace Safety is Non-Negotiable for Warehouse Operations" - XS Square Technologies discusses the importance of workplace safety for warehouse operations and offers solutions to eliminate workplace accidents. The article highlights the challenges faced by warehouses and the importance of implementing technology and warehouse automation to improve safety. It also discusses the impact of automation on job satisfaction and the need for ongoing training and collaboration with a provider of intelligent warehouse solutions. Finally, the article emphasizes the need for continuous improvement and recognition of human interve

### Test LM's ability to summarize 10 random samples of "Long text" - Observations
- Add task prompt to return output in English
- Word counts of "failed" cases not extreme - Stick to original prompt structure
- Improve prompt to extract "hiring context" - Future works

In [ ]:
# Implementation script to generate "text_summary" column
def summarize_text(df):
  summary_list = []
  for webtext in tqdm(df["text"]):
    messages = [
        {"role": "Act as a professional unstructured text summarizer",
        "content": f"Summarize website text in English and in strictly less than 100 words: {webtext}"}
    ]
    with torch.no_grad():  # Prevents unnecessary memory usage
      _, summary = complete(messages)
    summary_list.append(summary)
    torch.cuda.empty_cache()  # Clears unused memory
  df["text_summary"] = summary_list
  return df

# Apply LM summarization algo
df = summarize_text(df)

  0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
# (draft) save to csv
df_path = '/content/drive/MyDrive/Colab Notebooks/Interview_25/data/webtext_summary.csv'
df.to_csv(df_path, index=False)

In [ ]:
# print 10 samples for summarized text
samples = df.sample(10)
for text in samples["text_summary"]:
  print(text)
  print("---")
  print()

Keppel Seghers is an engineering firm located in Singapore and香港 with offices in Hong Kong and Doha. They offer services for construction, manufacturing, and infrastructure projects across Asia and the Middle East. Their team includes engineers, project managers, and designers who work together to deliver high-quality solutions. To get in touch with them, please visit their website or email them at info_keppelseghers keppel.com.
---

Choon Hin Group Company is an industrial company located in commercial residential, industrial, hospital, and facilities sectors. The company has a history dating back to 2019 when it was established. Board of Directors is responsible for overseeing the organization and running the company. Projects include commercial residential, industrial, hospital, and facility developments. Contact information can be found on the company's website. For more information, visit our website or contact us at [phone number] or [email address].
---

Welcome to kennametal So

## Review text summaries in detail
### **Questions to consider:**
- Can text summaries **"separate"/differentiate** hiring firms?
- How better can I prompt the LM to generate highly separable summaries?

### **Observations/Issues**
- Non-english characters
- Summaries provide little or zero information/directions on whether the company is hiring or not.
- ChatGPT 4.0 returns a direct classification accuracy of 0.35, this indicates that the quality of text data might not be good enough for separation of classes.

### **Solutions to explore:**
- Design a better LM summarization prompt.
- Research on existing/current methodologies for effective text summarization (perhaps with some business web context) to deduce hiring capacity of firms.

## Research articles and notes
- https://medium.com/data-science/setting-up-a-text-summarisation-project-daae41a1aaa3
  - Abstractive text summarization over extractive summarization.
  - How do we evaluate quality of summarized text for NLP applications? *ROUGE score*.
  - Fine-tuned models trained mainly on summarizing news articles. Furthermore, to compute metrics for text summarization, summary labels must be provided - Not very useful for our use-case.

In [ ]:
path = 'data/webtext_summary.csv'
df = pd.read_csv(path)
df.head()

,index,text,innovative_label,word_count,text_class,text_summary
0,192,Expert Web Development and Digital Marketing i...,0,296,Short text,"Web development, e-commerce development, socia..."
1,3690,Home Oon Bazul Skip to content Search for Home...,0,1374,Long text,The website is focused on providing legal and ...
2,3356,Our Latest Projects Twenty20 Energy Home About...,0,1093,Long text,Twenty20 Energy is an Australian-based renewab...
3,2009,Literature Library Literature Library InVivos ...,0,444,Short text,InVivos is an online repository of literature ...
4,1514,Aviation Unitech About Unitech Our Story Certi...,0,248,Short text,Unitech offers lightning and surge protection ...


In [ ]:
# return sample df (10 hiring vs 10 non-hiring)
sample_1 = df[df['innovative_label'] == 1].sample(10)
sample_2 = df[df['innovative_label'] == 0].sample(10)
sample_3 = pd.concat([sample_1, sample_2])

# select text and text summaries
sample = sample_3[['text', 'text_summary', "innovative_label"]]

In [ ]:
len(sample)

20

In [ ]:
# save csv for review - Review on Excel.
review_path = 'data/sample_review.csv'
sample.to_csv(review_path, index=False)